# Correlation x Heatmap: Cross Features

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy import stats
import matplotlib.ticker as mticker

## Functions

In [ ]:
def get_stat_columns(df_sleep: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """Return a DataFrame with only columns that start with `prefix`."""
    cols = [c for c in df_sleep.columns if c.startswith(prefix)]
    if not cols:
        raise ValueError(f"No columns found starting with '{prefix}'. "
                         f"Check STAT_PREFIXES in the config section.")
    sub = df_sleep[cols].copy()
    # Strip the prefix from column names so the heatmap shows feature names only
    sub.columns = [c[len(prefix):] for c in sub.columns]
    return sub

# Add correlation coefficient to each scatter plot
def corr_annotation(x, y, method='pearson', **kwargs):
    # Drop NaN pairs
    mask = ~np.isnan(x) & ~np.isnan(y)
    if method == "pearson" :
        r, p = stats.pearsonr(x[mask], y[mask])
        
    elif method == "spearman":
        r, p = stats.spearmanr(x[mask], y[mask])
    
    # Format p-value
    if p < 0.001:
        p_str = 'p < 0.001'
    elif p < 0.01:
        p_str = 'p < 0.01'
    elif p < 0.05:
        p_str = 'p < 0.05'
    else:
        p_str = f'p = {p:.3f}'
    
    ax = plt.gca()
    if method == "pearson":
        ax.annotate(
            f'r = {r:.2f}\np = {p:.4f}\n{p_str}',
            xy=(0.1, 0.85),
            xycoords='axes fraction',
            fontsize=8,
            color='red' if p < 0.05 else 'grey'  # red if significant
        )
    elif method == "spearman":
        ax.annotate(
            f'ρ = {r:.2f}\np = {p:.4f}\n{p_str}',
            xy=(0.1, 0.85),
            xycoords='axes fraction',
            fontsize=8,
            color='red' if p < 0.05 else 'grey'  # red if significant
        )
def seconds_to_hhmm(seconds, pos=None):
    seconds = int(seconds)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    return f"{hours:02d}:{minutes:02d}"

def seconds_to_clock_hhmm(seconds, pos=None):
    seconds = int(seconds) % (24 * 3600)
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    return f"{hours:02d}:{minutes:02d}"


## Get & Process Data

In [ ]:
# Example data
df_sleep = pd.read_csv('../../output/1_feature_extraction/df_features_sleep_2026-07-08.csv')
display(df_sleep.columns)

df_sleep_stages_stats = pd.read_csv('../../output/1_feature_extraction/df_features_sleep_stages_2026-07-08.csv')

display(df_sleep_stages_stats.columns)

df_sleep = df_sleep.merge(df_sleep_stages_stats, on='study_id', how='outer')

# Convert time columns to seconds
time_columns = ['median_sleep_onset', 'median_midpoint', 'median_wakeup', 'mean_sleep_onset', 'mean_midpoint', 'mean_wakeup']

for col in time_columns:
    df_sleep[col + '_seconds'] = pd.to_datetime(df_sleep[col], format='mixed').dt.time.apply(
        lambda x: x.hour * 3600 + x.minute * 60 + x.second
    )

# Midnight crossover adjustment
df_sleep['median_sleep_onset_s_adj'] = df_sleep['median_sleep_onset_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)
df_sleep['median_midpoint_s_adj'] = df_sleep['median_midpoint_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)

df_sleep['mean_sleep_onset_s_adj'] = df_sleep['mean_sleep_onset_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)
df_sleep['mean_midpoint_s_adj'] = df_sleep['mean_midpoint_seconds'].apply(
    lambda x: x + 24 * 3600 if x < 12 * 3600 else x
)
#drop seconds time columns
df_sleep = df_sleep.drop(columns=['median_sleep_onset_seconds', 'median_midpoint_seconds', 'mean_sleep_onset_seconds', 'mean_midpoint_seconds'])


df_sleep = get_stat_columns(df_sleep, "median_")
df_sleep = df_sleep.select_dtypes(include="number")
display(df_sleep.head())

df_step = pd.read_csv('../../output/1_feature_extraction/df_features_step_2026-07-08.csv')
display(df_step.columns)

df_step = df_step[df_step['study_id']!="DEC_46"]


df_step = get_stat_columns(df_step, "median_")
df_step = df_step.select_dtypes(include="number")
display(df_step.head())

# BBI
df_hr = pd.read_csv('../../output/1_feature_extraction/df_features_hr_2026-07-08.csv')
df_nocturnal = pd.read_csv('../../output/1_feature_extraction/df_features_nocturnal_hr_2026-07-08.csv')

df_hr = df_hr.merge(df_nocturnal, on='study_id', how='outer', suffixes=('', '_nocturnal'))
display(df_hr.columns)

#!drop rows where n_nights < 7
df_hr = df_hr[df_hr['n_nights'] >= 7]
display(df_hr)

df_hr = get_stat_columns(df_hr, "mean_")
df_hr = df_hr.select_dtypes(include="number")
display(df_hr.head())



label_map_hr = {
    # 'steps':'Steps',
    'hr': 'HR [bpm]',
    'rmssd': 'HRV [ms]',
    'hr_nocturnal': 'Nocturnal HR [bpm]',
    'rmssd_nocturnal': 'Nocturnal HRV [ms]',
}
#keep only columns that are in label_map
df_hr = df_hr[[col for col in df_hr.columns if col in label_map_hr.keys()]]
display(df_hr.columns)





# Sleep & Activity

In [ ]:
df_1 = pd.merge(df_sleep, df_step, left_index=True, right_index=True, how='outer')


label_map = {
    'sleep_duration':'Sleep [h]',
    'sleep_onset_s_adj': 'Sleep Onset',
    'wakeup_seconds': 'Wakeup Time',
    'midpoint_s_adj': 'Sleep Midpoint',
    'waso': 'WASO [min]',
    'awake': 'WASO Count',
    'ser': 'SER [%]',
    'light_sleep_duration': 'Light Sleep [h]',
    'deep_sleep_duration': 'Deep Sleep [h]',
    'rem_sleep_duration': 'REM Sleep [h]',
    'overall_sleep_score':'Overall Sleep Score',
    'steps': 'Steps',
    'sedentary_time_h': 'Sedentary Time [h]',
    'movement_time_h': 'Movement Time [h]',
    'ratio': 'Movement/Sedentary Ratio',
    'steps_2h': 'Steps 2h after Wakeup',
    'steps_4h': 'Steps 4h after Wakeup',
    'steps_onset_2h': 'Steps 2h before Onset',
    'steps_onset_4h': 'Steps 4h before Onset',
}
df_plot_1 = df_1.rename(columns=label_map)

display(df_plot_1.columns)

corr_spearman = df_plot_1.corr(method="spearman")

display(corr_spearman)
display(corr_spearman.columns)

# get number of features and participants
num_features = len(df_plot_1.columns)
num_participants = len(df_plot_1)
print(f"Number of features: {num_features}, Number of participants: {num_participants}")


In [ ]:
norm = Normalize(vmin=-1, vmax=1)
cmap = plt.cm.coolwarm

def corr_cell(x, y, **kwargs):
    ax = plt.gca()

    # Get variable names from the Series passed by PairGrid
    r = corr_spearman.loc[y.name, x.name]

    ax.set_facecolor(cmap(norm(r)))
    ax.annotate(
        f"{r:.2f}",
        xy=(0.5, 0.5),
        xycoords=ax.transAxes,
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
    )

    # Remove ticks inside heatmap cells
    ax.tick_params(
        axis="both",
        which="both",
        bottom=False,
        left=False,
        labelbottom=False,
        labelleft=False,
    )


sns.set_theme(style="white")

g = sns.PairGrid(df_plot_1, diag_sharey=False)

# Lower triangle: pair plot-like scatterplots
g.map_lower(sns.scatterplot, s=25, edgecolor="none")

# Diagonal: distributions
g.map_diag(sns.histplot, kde=True)

# Upper triangle: heatmap-like correlation cells
g.map_upper(corr_cell)


sleep_col = ["Sleep Onset", "Wakeup Time"]
sleep_midpoint_col = "Sleep Midpoint"


# Format axes for time variables and diagonal labels
for i, y_var in enumerate(g.y_vars):
    for j, x_var in enumerate(g.x_vars):
        ax = g.axes[i, j]

        if x_var in sleep_col:
            ax.xaxis.set_major_formatter(
                mticker.FuncFormatter(seconds_to_hhmm)
            )
        elif x_var == sleep_midpoint_col:
            ax.xaxis.set_major_formatter(
                mticker.FuncFormatter(seconds_to_clock_hhmm)
            )

        if y_var in sleep_col:
            ax.yaxis.set_major_formatter(
                mticker.FuncFormatter(seconds_to_hhmm)
            )
        elif y_var == sleep_midpoint_col:
            ax.yaxis.set_major_formatter(
                mticker.FuncFormatter(seconds_to_clock_hhmm)
            )
        if i == j:
            ax.tick_params(axis="y", labelleft=False)
            ax.set_ylabel("")

# add a colorbar for the correlations
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

g.figure.subplots_adjust(right=0.9)
cbar_ax = g.figure.add_axes([0.92, 0.15, 0.02, 0.7])
g.figure.colorbar(sm, cax=cbar_ax, label="Spearman $r_s$")
g.map_lower(corr_annotation, method='spearman')

g.figure.suptitle("Sleep & Activity Features: Pairwise Relationships & Spearman Correlations", fontsize=20, y=0.99)
g.figure.text(
    0.5, 0.97,
    "{} participants,{} features".format(num_participants, num_features),
    ha="center",
    va="top",
    fontsize=18
)

g.figure.subplots_adjust(top=0.95)


plt.show()

# Sleep & HR/HRV

In [ ]:
df_2 = pd.merge(df_sleep, df_hr, left_index=True, right_index=True, how='outer')


label_map = {
    'sleep_duration':'Sleep [h]',
    'sleep_onset_s_adj': 'Sleep Onset',
    'wakeup_seconds': 'Wakeup Time',
    'midpoint_s_adj': 'Sleep Midpoint',
    'waso': 'WASO [min]',
    'awake': 'WASO Count',
    'ser': 'SER [%]',
    'light_sleep_duration': 'Light Sleep [h]',
    'deep_sleep_duration': 'Deep Sleep [h]',
    'rem_sleep_duration': 'REM Sleep [h]',
    'overall_sleep_score':'Overall Sleep Score',
    'hr': 'HR [bpm]',
    'rmssd': 'HRV [ms]',
    'hr_nocturnal': 'Nocturnal HR [bpm]',
    'rmssd_nocturnal': 'Nocturnal HRV [ms]',
}
df_plot_2 = df_2.rename(columns=label_map)

display(df_plot_2.columns)

corr_spearman = df_plot_2.corr(method="spearman")

display(corr_spearman)
display(corr_spearman.columns)

# get number of features and participants
num_features = len(df_plot_2.columns)
num_participants = len(df_plot_2)
print(f"Number of features: {num_features}, Number of participants: {num_participants}")


In [ ]:
norm = Normalize(vmin=-1, vmax=1)
cmap = plt.cm.coolwarm

def corr_cell(x, y, **kwargs):
    ax = plt.gca()

    # Get variable names from the Series passed by PairGrid
    r = corr_spearman.loc[y.name, x.name]

    ax.set_facecolor(cmap(norm(r)))
    ax.annotate(
        f"{r:.2f}",
        xy=(0.5, 0.5),
        xycoords=ax.transAxes,
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
    )

    # Remove ticks inside heatmap cells
    ax.tick_params(
        axis="both",
        which="both",
        bottom=False,
        left=False,
        labelbottom=False,
        labelleft=False,
    )


sns.set_theme(style="white")

g = sns.PairGrid(df_plot_2, diag_sharey=False)

# Lower triangle: pair plot-like scatterplots
g.map_lower(sns.scatterplot, s=25, edgecolor="none")

# Diagonal: distributions
g.map_diag(sns.histplot, kde=True)

# Upper triangle: heatmap-like correlation cells
g.map_upper(corr_cell)


sleep_col = ["Sleep Onset", "Wakeup Time"]
sleep_midpoint_col = "Sleep Midpoint"


# Format axes for time variables and diagonal labels
for i, y_var in enumerate(g.y_vars):
    for j, x_var in enumerate(g.x_vars):
        ax = g.axes[i, j]

        if x_var in sleep_col:
            ax.xaxis.set_major_formatter(
                mticker.FuncFormatter(seconds_to_hhmm)
            )
        elif x_var == sleep_midpoint_col:
            ax.xaxis.set_major_formatter(
                mticker.FuncFormatter(seconds_to_clock_hhmm)
            )

        if y_var in sleep_col:
            ax.yaxis.set_major_formatter(
                mticker.FuncFormatter(seconds_to_hhmm)
            )
        elif y_var == sleep_midpoint_col:
            ax.yaxis.set_major_formatter(
                mticker.FuncFormatter(seconds_to_clock_hhmm)
            )
        if i == j:
            ax.tick_params(axis="y", labelleft=False)
            ax.set_ylabel("")

# add a colorbar for the correlations
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

g.figure.subplots_adjust(right=0.9)
cbar_ax = g.figure.add_axes([0.92, 0.15, 0.02, 0.7])
g.figure.colorbar(sm, cax=cbar_ax, label="Spearman $r_s$")
g.map_lower(corr_annotation, method='spearman')

g.figure.suptitle("Sleep & HR/HRV Features: Pairwise Relationships & Spearman Correlations", fontsize=20, y=0.99)
g.figure.text(
    0.5, 0.97,
    "{} participants,{} features".format(num_participants, num_features),
    ha="center",
    va="top",
    fontsize=18
)

g.figure.subplots_adjust(top=0.95)


plt.show()

# Activity & HR/HRV

In [ ]:
df_3 = pd.merge(df_step, df_hr, left_index=True, right_index=True, how='outer')


label_map = {
    'steps': 'Steps',
    'sedentary_time_h': 'Sedentary Time [h]',
    'movement_time_h': 'Movement Time [h]',
    'ratio': 'Movement/Sedentary Ratio',
    'steps_2h': 'Steps 2h after Wakeup',
    'steps_4h': 'Steps 4h after Wakeup',
    'steps_onset_2h': 'Steps 2h before Onset',
    'steps_onset_4h': 'Steps 4h before Onset',
    'hr': 'HR [bpm]',
    'rmssd': 'HRV [ms]',
    'hr_nocturnal': 'Nocturnal HR [bpm]',
    'rmssd_nocturnal': 'Nocturnal HRV [ms]',
}
df_plot_3 = df_3.rename(columns=label_map)

display(df_plot_3.columns)

corr_spearman = df_plot_3.corr(method="spearman")

display(corr_spearman)
display(corr_spearman.columns)

# get number of features and participants
num_features = len(df_plot_3.columns)
num_participants = len(df_plot_3)
print(f"Number of features: {num_features}, Number of participants: {num_participants}")


In [ ]:
norm = Normalize(vmin=-1, vmax=1)
cmap = plt.cm.coolwarm

def corr_cell(x, y, **kwargs):
    ax = plt.gca()

    # Get variable names from the Series passed by PairGrid
    r = corr_spearman.loc[y.name, x.name]

    ax.set_facecolor(cmap(norm(r)))
    ax.annotate(
        f"{r:.2f}",
        xy=(0.5, 0.5),
        xycoords=ax.transAxes,
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
    )

    # Remove ticks inside heatmap cells
    ax.tick_params(
        axis="both",
        which="both",
        bottom=False,
        left=False,
        labelbottom=False,
        labelleft=False,
    )


sns.set_theme(style="white")

g = sns.PairGrid(df_plot_3, diag_sharey=False)

# Lower triangle: pair plot-like scatterplots
g.map_lower(sns.scatterplot, s=25, edgecolor="none")

# Diagonal: distributions
g.map_diag(sns.histplot, kde=True)

# Upper triangle: heatmap-like correlation cells
g.map_upper(corr_cell)


sleep_col = ["Sleep Onset", "Wakeup Time"]
sleep_midpoint_col = "Sleep Midpoint"


# Format axes for time variables and diagonal labels
for i, y_var in enumerate(g.y_vars):
    for j, x_var in enumerate(g.x_vars):
        ax = g.axes[i, j]
        if i == j:
            ax.tick_params(axis="y", labelleft=False)
            ax.set_ylabel("")

# add a colorbar for the correlations
sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

g.figure.subplots_adjust(right=0.9)
cbar_ax = g.figure.add_axes([0.92, 0.15, 0.02, 0.7])
g.figure.colorbar(sm, cax=cbar_ax, label="Spearman $r_s$")
g.map_lower(corr_annotation, method='spearman')

g.figure.suptitle("Activity & HR/HRV Features: Pairwise Relationships & Spearman Correlations", fontsize=20, y=0.99)
g.figure.text(
    0.5, 0.97,
    "{} participants,{} features".format(num_participants, num_features),
    ha="center",
    va="top",
    fontsize=18
)

g.figure.subplots_adjust(top=0.95)


plt.show()